In [13]:
# Regrid BMR and ozone data to match population grid

In [1]:
import os
import xarray as xr
import regionmask
import matplotlib.pyplot as plt
import numpy as np

In [2]:
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
population = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_2000-2100.nc")

pop_lat = population.lat.values
pop_lon = population.lon.values

In [3]:
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
bmr = xr.open_dataarray(f"{BMR_DIR}GBD_BMR_Country_Mask_COPD_1990-2009.nc")

bmr_lat = bmr.lat.values
bmr_lon = bmr.lon.values

bmr_lat_mask = xr.ufuncs.logical_and(bmr_lat <= pop_lat.max(), bmr_lat >= pop_lat.min())
bmr_lon_mask = xr.ufuncs.logical_and(bmr_lon <= pop_lon.max(), bmr_lon >= pop_lon.min())

new_bmr = bmr.sel(lat=bmr_lat_mask)  # longitudes are the same
bmr_interp = new_bmr.interp(lat=population.lat, lon=population.lon, method="linear")

bmr_interp.to_netcdf(f"{BMR_DIR}GBD_BMR_Country_Mask_COPD_popgrid_1990-2009.nc")

In [ ]:
# ensemble numbers
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"
scenarios = ["ARISE", "SSP245"]

for scenario in scenarios:
    for ens_num in range(2, 11):
        print(f"Processing {scenario} ensemble number {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2069"
        elif scenario == "SSP245":
            dates = "2020-2069"
        o3 = xr.open_dataset(f"{O3_DIR}OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

        o3_lat = o3.lat.values
        o3_lon = o3.lon.values

        o3_lat_mask = xr.ufuncs.logical_and(o3_lat <= pop_lat.max(), o3_lat >= pop_lat.min())
        o3_lon_mask = xr.ufuncs.logical_and(o3_lon <= pop_lon.max(), o3_lon >= pop_lon.min())

        new_o3 = o3.sel(lat=o3_lat_mask)  # longitudes are the same
        o3_interp = new_o3.interp(lat=population.lat, lon=population.lon, method="linear")

        print("Saving regridding o3")
        o3_interp.to_netcdf(f"{O3_DIR}OSDMA8_BC_popgrid_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

print("Processing complete")

Processing ARISE ensemble number 02
Saving regridding o3
Processing ARISE ensemble number 03
Saving regridding o3
Processing ARISE ensemble number 04
Saving regridding o3
Processing ARISE ensemble number 05
Saving regridding o3
Processing ARISE ensemble number 06
Saving regridding o3
Processing ARISE ensemble number 07
Saving regridding o3
Processing ARISE ensemble number 08
